# 22. Baseline — EmbDI

EmbDI(Cappuzzo et al., SIGMOD 2020)의 그래프 구성(행 `RID_`·값 `VAL_`·컬럼 `COL_` 노드)을 따라 테이블을 그래프로 만들고,
Word2Vec(gensim)으로 노드 임베딩을 학습한 뒤 서로 다른 테이블의 컬럼 노드 간 코사인 유사도로 FK 후보를 순위화한다.
원본 EmbDI 패키지를 쓰지 않은 단순화 구현으로, random walk 대신 그래프의 간선을 그대로 학습 문장으로 사용한다.
원본 코드: https://gitlab.eurecom.fr/cappuzzo/embdi (Apache-2.0)

- 출력: `workspace/embdi_ranking_metrics_<db>[_mask].json`

In [ ]:
# !pip install gensim matplotlib pandas numpy scikit-learn tqdm

1. 환경 설정 및 라이브러리 임포트
실험에 필요한 라이브러리를 로드합니다. gensim은 Word2Vec 학습을 위해, tqdm은 대용량 데이터 처리 시 진행 상황을 보기 위해 사용합니다.

In [ ]:
import os
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve
from itertools import combinations
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

db_name = "superhero"
db_name = "california_schools"
db_name = "toxicology"
db_name = "thrombosis_prediction"
# db_name = "debit_card_specializing"
# db_name = "total_test"
# db_name = "card_games"
# db_name = "european_football_2"
# db_name = "codebase_community"
# db_name = "superhero_mask"
# db_name = "toxicology_mask"
# db_name = "thrombosis_prediction_mask"
# db_name = "california_schools_mask"

# 경로 설정 
DATA_DIR = f"../workspace/dataset_mini/bird_data_tables/{db_name}"
GROUND_TRUTH_PATH = f"../workspace/dataset_mini/bird_data_tables/{db_name}/ground_truth.json"


2. 데이터 로더 (Data Loader)
BIRD 폴더 내의 모든 CSV 파일을 읽어와 딕셔너리 형태로 저장합니다. 메모리 문제가 발생할 경우 샘플링 로직을 추가할 수 있습니다.

In [ ]:
def load_bird_csvs(data_dir):
    dfs = {}
    csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
    print(f"Found {len(csv_files)} tables. Loading...")
    
    for file in tqdm(csv_files, desc="Loading CSVs"):
        table_name = file.replace('.csv', '')
        try:
            # BIRD 데이터셋은 인코딩 문제가 있을 수 있어 latin1 혹은 utf-8-sig 고려
            df = pd.read_csv(os.path.join(data_dir, file), low_memory=False)
            dfs[table_name] = df
        except Exception as e:
            print(f"Error loading {file}: {e}")
    return dfs

all_dfs = load_bird_csvs(DATA_DIR)

3. EmbDI 그래프 생성 (Edge List Construction)
EmbDI의 핵심은 테이블, 컬럼, 값을 하나의 그래프로 엮는 것입니다. 여기서는 (Row-Value)와 (Column-Value)를 연결하는 에지 리스트를 생성합니다.

In [ ]:
def build_embdi_edges(dfs):
    edges = []
    for table_name, df in tqdm(dfs.items(), desc="Building Graph Edges"):
        # 컬럼 노드 생성
        cols = df.columns
        for idx, row in df.iterrows():
            row_node = f"RID_{table_name}_{idx}"
            for col in cols:
                col_node = f"COL_{table_name}_{col}"
                val = str(row[col]).strip()
                
                # 결측치 제외 및 노드 연결
                if val and val.lower() != 'nan':
                    val_node = f"VAL_{val}"
                    edges.append([row_node, val_node]) # Row - Value 연결
                    edges.append([col_node, val_node]) # Column - Value 연결
    return edges

edgelist = build_embdi_edges(all_dfs)
print(f"Total Edges created: {len(edgelist)}")

4. Word2Vec 학습 (Representation Learning)
생성된 에지 리스트를 바탕으로 각 노드(컬럼 등)를 벡터 공간에 투영합니다.

In [ ]:
# Word2Vec 학습 설정
# window=1은 에지로 직접 연결된 노드 간의 관계를 강조합니다.
print("Training Word2Vec for EmbDI...")
model = Word2Vec(
    sentences=edgelist, 
    vector_size=128, 
    window=1, 
    min_count=1, 
    workers=8, 
    epochs=5
)

# 학습 결과 확인 (예시: 특정 컬럼과 유사한 노드 찾기)
sample_col = f"COL_{list(all_dfs.keys())[0]}_{all_dfs[list(all_dfs.keys())[0]].columns[0]}"
print(f"Most similar to {sample_col}:")
print(model.wv.most_similar(sample_col)[:5])

5. 성능 평가 (Evaluation)
Ground Truth와 비교하여 최종 지표를 산출합니다. 모든 컬럼 쌍을 비교하되, 동일 테이블 내의 매칭은 제외합니다.

In [ ]:
def run_evaluation(model, dfs, gt_json_path, threshold=0.98):
# def run_evaluation(model, dfs, gt_json_path, threshold=0.90):
    import json
    from itertools import combinations
    from tqdm.auto import tqdm
    from sklearn.metrics import precision_score, recall_score, f1_score

    # 1. Ground Truth 로드 및 정규화
    with open(gt_json_path, 'r') as f:
        gt_data = json.load(f)
    
    gt_set = set()
    for pair_list in gt_data:
        if len(pair_list) < 2: continue
        part1, part2 = pair_list[0].split('.'), pair_list[1].split('.')
        if len(part1) == 2 and len(part2) == 2:
            pair = tuple(sorted([f"COL_{part1[0]}_{part1[1]}", f"COL_{part2[0]}_{part2[1]}"]))
            gt_set.add(pair)

    # 2. 모델 내 컬럼 노드 추출
    all_col_nodes = [node for node in model.wv.index_to_key if node.startswith('COL_')]

    if not all_col_nodes:
        print("⚠️  Warning: 모델에 COL_ 노드가 없습니다. Word2Vec 학습이 완료됐는지 확인하세요.")
        return 0.0, 0.0, 0.0, [], [], 0, 0, 0, 0, 0, 0

    # 3. 컬럼 노드 → 테이블 이름 매핑 (테이블명 자체에 '_'가 포함된 경우 대응)
    col_to_table = {}
    for table_name in dfs.keys():
        prefix = f"COL_{table_name}_"
        for node in all_col_nodes:
            if node.startswith(prefix):
                col_to_table[node] = table_name

    predicted_log = []
    gt_scores_log = []
    y_true, y_scores = [], []
    
    skipped_same_table = 0
    skipped_keyerror = 0
    
    # 4. 모든 컬럼 쌍 유사도 계산
    for c1, c2 in tqdm(combinations(all_col_nodes, 2), desc="Evaluating Pairs"):
        t1 = col_to_table.get(c1, '')
        t2 = col_to_table.get(c2, '')
        if t1 == t2:
            skipped_same_table += 1
            continue  # 동일 테이블 제외
        
        current_pair = tuple(sorted([c1, c2]))
        is_gt_member = current_pair in gt_set
        
        try:
            sim = model.wv.similarity(c1, c2)
            y_scores.append(sim)
            y_true.append(1 if is_gt_member else 0)
            
            if sim >= threshold:
                predicted_log.append({
                    'pair': current_pair,
                    'similarity': sim,
                    'is_correct': "✅ MATCH" if is_gt_member else "❌ FALSE POSITIVE"
                })
            
            if is_gt_member:
                gt_scores_log.append({
                    'pair': current_pair,
                    'similarity': sim
                })
                
        except KeyError:
            skipped_keyerror += 1
            y_true.append(1 if is_gt_member else 0)
            y_scores.append(0.0)
            
            if is_gt_member:
                gt_scores_log.append({
                    'pair': current_pair,
                    'similarity': "NOT_IN_MODEL (KeyError)"
                })
            continue

    # 5. 결과 출력 1: Threshold를 넘은 상위 예측들 (오답 분석용)
    print(f"\n" + "="*60)
    print(f" [REPORT 1] Top Predictions (Sim >= {threshold})")
    print(f" (Showing how the model is thinking - including False Positives)")
    print("="*60)
    predicted_log = sorted(predicted_log, key=lambda x: x['similarity'] if isinstance(x['similarity'], float) else -1, reverse=True)
    for res in predicted_log[:30]: 
        print(f"[{res['is_correct']}] {res['pair'][0]} <-> {res['pair'][1]} | Sim: {res['similarity']:.4f}")

    # 6. 결과 출력 2: Ground Truth에 명시된 정답들의 실제 점수 (미탐 분석용)
    print(f"\n" + "="*60)
    print(f" [REPORT 2] Ground Truth Pair Scores")
    print(f" (Checking why the model missed these or how high they scored)")
    print("="*60)
    for res in gt_scores_log:
        sim_val = f"{res['similarity']:.4f}" if isinstance(res['similarity'], float) else res['similarity']
        status = "🚩 BELOW THRESHOLD" if isinstance(res['similarity'], float) and res['similarity'] < threshold else "🎯 FOUND"
        if sim_val == "NOT_IN_MODEL (KeyError)": status = "⚠️ MISSING NODE"
        print(f"[{status}] {res['pair'][0]} <-> {res['pair'][1]} | Sim: {sim_val}")

    # 7. 지표 계산
    if len(y_true) == 0:
        print("⚠️  Warning: 크로스 테이블 평가 쌍이 없습니다. 테이블이 1개뿐이거나 모든 컬럼이 동일 테이블에 속합니다.")
        return 0.0, 0.0, 0.0, [], [], 0, 0, 0, 0, skipped_same_table, skipped_keyerror

    y_pred = [1 if s >= threshold else 0 for s in y_scores]

    tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 1)
    fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 1)
    tn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 0)
    fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 0)

    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return p, r, f1, y_true, y_scores, tp, fp, tn, fn, skipped_same_table, skipped_keyerror

precision, recall, f1, y_true, y_scores, tp, fp, tn, fn, skipped_same_table, skipped_keyerror = run_evaluation(model, all_dfs, GROUND_TRUTH_PATH)

print(f"\n[Pair Statistics]")
print(f"  Total combinations:     {skipped_same_table + len(y_true)}")
print(f"  Same-table excluded:    {skipped_same_table}")
print(f"  KeyError (sim=0):       {skipped_keyerror}")
print(f"  Evaluated pairs:        {len(y_true)}")

print(f"\n[Confusion Matrix]")
print(f"  TP (True Positive):   {tp}")
print(f"  FP (False Positive):  {fp}")
print(f"  TN (True Negative):   {tn}")
print(f"  FN (False Negative):  {fn}")

print(f"\n[Performance Metrics]")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

6. Precision-Recall Curve 시각화
Threshold 0.5가 최적인지 확인하기 위해 곡선을 그려봅니다. CRIM 모델과 비교할 때 이 그래프의 면적(AUC)을 비교하면 아주 훌륭한 분석 자료가 됩니다.

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_true, y_scores)

plt.figure(figsize=(8, 6))
plt.plot(recalls, precisions, label='EmbDI (Word2Vec)')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for BIRD Dataset')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ── PR-AUC + Recall@K (CRIM / Valentine / DeepJoin 과 동일 프로토콜) ────
# 임계값-독립 ranking 지표로 4개 모델을 한 표에 놓고 비교
#   PR-AUC          : 극도로 불균형한 FK 탐지에 적합
#   Recall@K (K=|GT|): Valentine SIGMOD 2021 표준 (K=|GT| → P@K=R@K=F1@K)
#
# EmbDI는 이미 run_evaluation 에서 cross-table 쌍에 대해
# y_true / y_scores 를 산출함 (sorted tuple = frozenset 과 동치).
# 따라서 그대로 사용해서 ranking 지표 계산.

from sklearn.metrics import average_precision_score
import numpy as np
import json
import os

# ── 0. 입력 확인 ───────────────────────────────────────────────
# 직전 셀에서 y_true, y_scores 가 만들어져 있어야 함
y_true_arr  = np.array(y_true,  dtype=np.int32)
y_score_arr = np.array(y_scores, dtype=np.float32)

total_pairs = len(y_true_arr)
K = int(y_true_arr.sum())  # 평가 공간 내 GT 개수

print(f"\n{'='*70}")
print(f"[EmbDI 임계값-독립 평가 — CRIM 호환 프로토콜]")
print(f"  DB: {db_name}")
print(f"{'='*70}")
print(f"  평가 후보쌍(cross-table) : {total_pairs}")
print(f"  GT 쌍 (평가공간 내)      : {K}")
print(f"  same-table 제외          : {skipped_same_table}")
print(f"  Model에 없는 노드 (sim=0): {skipped_keyerror}")

# ── 1. PR-AUC ──────────────────────────────────────────────────
pr_auc = (average_precision_score(y_true_arr, y_score_arr)
          if K > 0 else float('nan'))

# ── 2. Recall@K (K = |GT|) ────────────────────────────────────
if K > 0 and K <= total_pairs:
    top_k_idx = np.argsort(-y_score_arr, kind='stable')[:K]
    tp_at_K = int(y_true_arr[top_k_idx].sum())
    recall_at_K = tp_at_K / K
else:
    tp_at_K, recall_at_K = 0, float('nan')

# ── 3. Recall@3K (보조) ───────────────────────────────────────
if K > 0 and 3*K <= total_pairs:
    top_3k_idx = np.argsort(-y_score_arr, kind='stable')[:3*K]
    recall_at_3K = int(y_true_arr[top_3k_idx].sum()) / K
else:
    recall_at_3K = float('nan')

print(f"\n  PR-AUC         : {pr_auc:.4f}")
print(f"  Recall@{K:<3}     : {recall_at_K:.4f}  ({tp_at_K}/{K})")
if not np.isnan(recall_at_3K):
    print(f"  Recall@{3*K:<3}     : {recall_at_3K:.4f}")

# ── 4. 참고: 기존 threshold F1 ─────────────────────────────────
print(f"\n  [참고 — 기존 threshold F1]")
print(f"    Precision: {precision:.4f}")
print(f"    Recall   : {recall:.4f}")
print(f"    F1       : {f1:.4f}")

# ── 5. 저장 (CRIM / Valentine / DeepJoin 과 동일 스키마) ──────────
embdi_ranking_path = (
    f'../workspace/'
    f'embdi_ranking_metrics_{db_name}.json'
)
os.makedirs(os.path.dirname(embdi_ranking_path), exist_ok=True)
with open(embdi_ranking_path, 'w') as f:
    json.dump({
        'db':          db_name,
        'model':       'EmbDI_Word2Vec',
        'n_gt':        K,
        'total_pairs': total_pairs,
        'pr_auc':       round(float(pr_auc), 4),
        'recall_at_K':  round(float(recall_at_K), 4),
        'recall_at_3K': (round(float(recall_at_3K), 4)
                         if not np.isnan(recall_at_3K) else None),
        'tp_at_K':      tp_at_K,
        'f1_at_threshold':        round(float(f1), 4),
        'precision_at_threshold': round(float(precision), 4),
        'recall_at_threshold':    round(float(recall), 4),
        'skipped_same_table':     skipped_same_table,
        'skipped_keyerror':       skipped_keyerror,
        # PR curve 그리기용 raw 데이터
        'y_true':                 y_true_arr.tolist(),
        'y_score':                [round(float(s), 6) for s in y_score_arr],
        'note': ('Pair identity = sorted(c1, c2) ≡ frozenset; '
                 'compatible with CRIM/Valentine/DeepJoin undirected protocol.'),
    }, f, indent=2)
print(f"\n[저장] {embdi_ranking_path}")


In [ ]:
# ── Classic metrics (TP/FP/FN/TN, Precision/Recall/F1) 저장 ─────────
# run_evaluation 의 직접 반환값을 _classic 파일로 별도 저장.

import json
import os

classic_path = (
    f'../workspace/'
    f'embdi_ranking_metrics_{db_name}_classic.json'
)
os.makedirs(os.path.dirname(classic_path), exist_ok=True)

classic_data = {
    'db':                  db_name,
    'model':               'EmbDI_Word2Vec',
    # run_evaluation 의 기본 threshold (변경됐을 수 있으니 변수에서 읽음)
    'threshold':           float(globals().get('THRESHOLD', 0.98)),
    'tp':                  int(tp),
    'fp':                  int(fp),
    'fn':                  int(fn),
    'tn':                  int(tn),
    'precision':           round(float(precision), 4),
    'recall':              round(float(recall), 4),
    'f1':                  round(float(f1), 4),
    'n_gt':                int(tp + fn),
    'total_pairs':         int(tp + fp + tn + fn),
    'skipped_same_table':  int(skipped_same_table),
    'skipped_keyerror':    int(skipped_keyerror),
}

with open(classic_path, 'w') as f:
    json.dump(classic_data, f, indent=2)
print(f"[저장] {classic_path}")